# Week 1 Lab — Know Your Data  
### Part 1: First Contact & Documentation Audit  
Dataset: UCI Credit Card Default (30,000 rows, 25 columns)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

df = pd.read_csv("UCI_Credit_Card.csv")
print(df.shape)                                           # (30000, 25)
print(df["default.payment.next.month"].mean().round(3))   # ~0.221 default rate

(30000, 25)
0.221


## 1.1 — Shape, Types, and Missing Values  
We begin with a basic profile of the dataset:  
- shape  
- data types  
- literal blanks  

In [2]:
# section1_profile.py
print(df.shape)
print(df.dtypes.value_counts())
print(df.isna().sum().sum())        # total blanks in the whole file

(30000, 25)
float64    13
int64      12
Name: count, dtype: int64
0


## 1.1 — Column Name Audit  
The repayment-status columns should represent six consecutive months.  
We check which columns exist and whether any month is missing.

In [3]:
print([c for c in df.columns if c.startswith("PAY_") and "AMT" not in c])

['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']


## 1.2 — Audit the Data Against Its Documentation  
We compare actual codes in the dataset with the official codebook  
(SEX, EDUCATION, MARRIAGE, PAY_0).


In [6]:
for c in ["SEX", "EDUCATION", "MARRIAGE"]:
    print(f"{c:10s} {sorted(df[c].unique().tolist())}")

print(f"{'PAY_0':10s} {sorted(df['PAY_0'].unique().tolist())}")

SEX        [1, 2]
EDUCATION  [0, 1, 2, 3, 4, 5, 6]
MARRIAGE   [0, 1, 2, 3]
PAY_0      [-2, -1, 0, 1, 2, 3, 4, 5, 6, 7, 8]


## Count how many rows contain undocumented codes  
This tells us how serious each documentation mismatch is.

In [7]:
print("EDUCATION in {0,5,6}:", df.EDUCATION.isin([0, 5, 6]).sum())
print("MARRIAGE == 0      :", (df.MARRIAGE == 0).sum())
print("PAY_0 in {-2, 0}   :", df.PAY_0.isin([-2, 0]).sum(),
      f"({100 * df.PAY_0.isin([-2, 0]).mean():.1f}%)")

EDUCATION in {0,5,6}: 345
MARRIAGE == 0      : 54
PAY_0 in {-2, 0}   : 17496 (58.3%)


## Behavioural Inference for Undocumented PAY_0 Codes  
We examine default rates for each PAY_0 value  
to infer what undocumented codes might represent.

In [8]:
rate = df.groupby("PAY_0")["default.payment.next.month"].agg(["mean", "size"])
rate.assign(mean=rate["mean"].round(3))

,mean,size
PAY_0,,
-2,0.132,2759
-1,0.168,5686
0,0.128,14737
1,0.339,3688
2,0.691,2667
3,0.758,322
4,0.684,76
5,0.500,26
6,0.545,11


## Interpretation  
- Codes −2 and 0 behave like “no delay” categories.  
- They default **less** than the documented “paid duly” code (−1).  
- They are **nothing like** a one‑month delay (1), which defaults at ~34%.

### Proposed meaning  
**−2 and 0 likely represent internal variants of “paid on time”.**

### Confidence  
Moderate.  
Behavioural inference is weaker than documentation because it assumes  
the relationship between repayment status and default risk is stable  
and monotonic — which may not be true.

#Checkpoint 1:

### 1. Shape & blanks  
- 30,000 rows  
- 25 columns  
- 0 literal blanks  

### 2. Undocumented codes  
- EDUCATION → 345 rows  
- MARRIAGE → 54 rows  
- PAY_0 → 17,496 rows (58.3%)  

### 3. Why PAY_0 is serious  
PAY_0 is the **most important behavioural feature**.  
If 58% of it is undocumented, your strongest predictor is partly undefined.  
EDUCATION’s undocumented codes affect only ~1% of rows and matter far less.
